# 00 — Setup, settings and test data

**Run this first.** Everything else depends on the files it builds.

### What this project is

A like-for-like timing comparison of two ways to process data files: **DuckDB**, which runs
inside a single process, and **Apache Spark**, which is built to spread work across many
machines.

> Spark is a **delivery company**. DuckDB is **your own kitchen**.
>
> For a big catered event you call the delivery company. For a sandwich you walk to the kitchen.
> Nobody thinks the kitchen replaces the caterers — but nobody orders a catering van for a
> sandwich either.

The question is not which is better. It is **which jobs sit on which side of the line**, and
that is a question with a measurable answer.

### The rules

1. **Same data.** Both engines read the same Parquet files from the same folder.
2. **Same names.** Both see tables called `sales`, `customers`, `products` and so on, so most
   cases send the *identical SQL string* to both. Where that is impossible the case is marked.
3. **Same budget.** Both engines get exactly the same memory and the same thread count. This
   is the most important fairness rule here.
4. **Warm-up first.** One untimed run before the clock starts, so we never time a cold JVM.
   That is generous to Spark on purpose.
5. **Startup excluded.** Spark's start-up time is measured and reported, but is *not* added to
   any benchmark. Every timing below is work only.
6. **Answers are checked.** A fast wrong answer is worthless, so both results are compared.
7. **Spark is tuned.** Every Spark setting we changed makes it *faster* than its defaults.
   They are all visible in `bench/config.py`.

### What this cannot show

Local Spark on one machine cannot demonstrate Spark's actual benefit, which is **many
machines**. So this measures the **cost** of distributing work, not the payoff. The payoff
appears above the sizes a single machine can hold — and notebook 09 finds where that is.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from bench import config as C, datagen, engines
import pandas as pd

print(C.summary())

## The settings

Everything that could affect a number lives in `bench/config.py`. Nothing is hidden anywhere
else.

If you think a setting is unfair to either engine, **change it there and re-run**. That is the
point of keeping it in one file.

The two Spark settings worth knowing about, because leaving them alone is what makes most
amateur benchmarks worthless:

* **`spark.sql.shuffle.partitions`** — Spark's default is 200. On one machine that means
  chopping every job into 200 pieces and spending all the time on paperwork. We set it to the
  core count.
* **`spark.sql.execution.arrow.pyspark.enabled`** — off by default. Without it we would be
  timing the handover of results to Python rather than timing Spark.

In [ ]:
print("SPARK SETTINGS IN FORCE (every one makes Spark faster than its defaults)\n")
for k, v in C.SPARK_CONF.items():
    print(f"   {k:<48} {v}")

## The test data

Three tables, shaped like data any business would have: one big table of events joined to a
couple of small lookups.

The `sales` table is deliberately **wide** — 22 columns — because real tables are wide, and it
lets us show that a good engine reads only the columns it needs.

It is also deliberately **dirty**. Data-quality checks on perfectly clean data measure nothing,
so we plant defects on purpose and list them below, so nobody has to guess what is in there.

In [ ]:
print("DEFECTS PLANTED ON PURPOSE\n")
for k, v in datagen.DEFECTS.items():
    print(f"   {v:<32} {k}")
print("\nBuilding datasets (this is the slow part; later notebooks reuse these files)...\n")

for n in C.SIZES:
    datagen.build(n, with_extras=(n == C.MAIN_SIZE))

print("\nOn disk:")
display(pd.DataFrame(datagen.disk_report()))

### Companion datasets

Alongside the three main tables we build a few extras so the later notebooks have something
realistic to work with:

| Dataset | Why it exists |
|---|---|
| `returns` | A **second large table**, so notebook 03 can do a genuine big-to-big shuffle join |
| `sales_v2` | A **later snapshot** with changes and deletions, for diffing and reconciliation |
| `sales_part` | A **partitioned** copy, to test skipping whole folders rather than filtering rows |
| `sales_small_*` | **200 small files**, for the compaction job every team has |
| `sales_*.csv` | A **CSV extract**, for the classic "convert this to Parquet" job |

In [ ]:
paths = datagen.paths(C.MAIN_SIZE)
duck  = engines.get_duckdb()
spark = engines.get_spark()
attached = engines.attach(duck, spark, paths)
print("Both engines can now see:", ", ".join(attached))
print()
print(engines.describe())

## A look at the data itself

In [ ]:
print("SALES — one row per purchase, 22 columns:")
display(duck.execute("SELECT * FROM sales LIMIT 5").df())

print("\nHow big is this really? Some context for the numbers that follow:")
n = C.MAIN_SIZE
print(f"   {C.human(n)} sales rows is roughly what a business doing 3,000 orders a day")
print(f"   would accumulate in about {n/3000/365:.1f} years.")

print("\nThe defects are really there:")
display(duck.execute("""
SELECT count(*)                                            AS total_rows,
       count(*) - count(customer_id)                       AS null_customer_id,
       count(*) - count(amount)                            AS null_amount,
       sum(CASE WHEN amount < 0 THEN 1 ELSE 0 END)         AS negative_amount
FROM sales
""").df())

display(duck.execute("""
SELECT country, count(*) AS customers FROM customers
GROUP BY 1 ORDER BY 1 LIMIT 8
""").df())
print("Note the same country appears several ways. Notebook 05 cleans that up.")
engines.stop_spark()

---

**Next:** notebook `01_scan_and_filter` starts the comparison. Each notebook covers one family
of work and saves its results to `results/`; notebook `10_summary` combines them all.